# SVM Classification of LA County Fire Susceptibility (0–4)

This notebook trains a **Support Vector Machine (SVM)** to classify locations in `LA_County_Fire_Susceptibility_0_to_4.kml` into the fire-susceptibility classes found in the file:

| Score | Class |
|---|---|
| 0 | Water |
| 1 | Low – developed / barren |
| 2 | Moderate – grass / crops / open space |
| 3 | High – shrub / scrub |
| 4 | Highest index – forest |

**About the data.** The KML does not contain point features. It holds 64 `Placemark`s (MultiPolygons), grouped into one folder per class, plus a county-boundary outline. Vertices of polygons sit on class borders and are shared between neighbouring classes, so they make poor labelled points. Instead, this notebook:

1. Parses the polygons (outer rings and holes) for each class.
2. Draws random points across the county and labels each by **which class polygon contains it**.
3. Trains an RBF-kernel SVM on `(longitude, latitude)` (converted to km) to predict the class.

> The score is an ordinal reclassification of NLCD 2024 land cover, not a calibrated fire probability. Because the only features are coordinates, the SVM learns the *spatial layout* of the classes. You could add features such as elevation, slope or distance to water to make this a richer model.

**Requirements:** `numpy`, `pandas`, `matplotlib`, `scikit-learn` (no network or GIS libraries needed).

In [ ]:
import re
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET
from matplotlib.path import Path
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, accuracy_score,
                             balanced_accuracy_score, cohen_kappa_score)

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

# ---- Settings you may want to change ----
KML_PATH = "LA_County_Fire_Susceptibility_0_to_4.kml"   # path to the uploaded file
N_CANDIDATES = 120_000   # random points drawn over the bounding box (only those inside a class polygon are kept)
MAX_PER_CLASS = 3_000    # cap per class to keep SVM training fast and balanced (set None to keep all)

## 1. Parse the KML into polygons per class

In [ ]:
KML_NS = {"k": "http://www.opengis.net/kml/2.2"}

def parse_coords(text):
    """'lon,lat,alt lon,lat,alt ...' -> (n, 2) array of lon/lat."""
    vals = np.array(re.split(r"[,\s]+", text.strip()), dtype=float)
    return vals.reshape(-1, 3)[:, :2]

def get_score(placemark):
    for sd in placemark.iter("{http://www.opengis.net/kml/2.2}SimpleData"):
        if sd.get("name") == "score":
            return int(sd.text)
    return None

tree = ET.parse(KML_PATH)
root = tree.getroot()

polys = []   # each: dict(score, outer (n,2), holes [ (m,2), ... ], bbox)
for pm in root.iter("{http://www.opengis.net/kml/2.2}Placemark"):
    score = get_score(pm)
    if score is None:            # county boundary outline has no score -> skip
        continue
    for poly in pm.iter("{http://www.opengis.net/kml/2.2}Polygon"):
        outer = poly.find("k:outerBoundaryIs/k:LinearRing/k:coordinates", KML_NS)
        if outer is None:
            continue
        outer_xy = parse_coords(outer.text)
        holes = [parse_coords(h.text) for h in
                 poly.findall("k:innerBoundaryIs/k:LinearRing/k:coordinates", KML_NS)]
        polys.append(dict(
            score=score, outer=outer_xy, holes=holes,
            bbox=(outer_xy[:, 0].min(), outer_xy[:, 1].min(),
                  outer_xy[:, 0].max(), outer_xy[:, 1].max())))

summary = (pd.DataFrame({"score": [p["score"] for p in polys],
                         "holes": [len(p["holes"]) for p in polys]})
           .groupby("score").agg(polygons=("holes", "size"), holes=("holes", "sum")))
print(f"{len(polys):,} polygons parsed")
summary

## 2. Build a labelled point dataset

Random points are drawn across the bounding box of the data. Each point is labelled by the class polygon that contains it (inside an outer ring and not inside a hole). Points that fall outside every polygon (e.g. outside the county) are discarded.

In [ ]:
all_xy = np.vstack([p["outer"] for p in polys])
lon_min, lat_min = all_xy.min(axis=0)
lon_max, lat_max = all_xy.max(axis=0)
print(f"Bounding box: lon [{lon_min:.3f}, {lon_max:.3f}], lat [{lat_min:.3f}, {lat_max:.3f}]")

cand = np.column_stack([rng.uniform(lon_min, lon_max, N_CANDIDATES),
                        rng.uniform(lat_min, lat_max, N_CANDIDATES)])
labels = np.full(N_CANDIDATES, -1, dtype=int)

t0 = time.time()
for p in polys:
    x0, y0, x1, y1 = p["bbox"]
    idx = np.where((cand[:, 0] >= x0) & (cand[:, 0] <= x1) &
                   (cand[:, 1] >= y0) & (cand[:, 1] <= y1) & (labels < 0))[0]
    if idx.size == 0:
        continue
    inside = Path(p["outer"]).contains_points(cand[idx])
    idx = idx[inside]
    for h in p["holes"]:
        if idx.size == 0:
            break
        hx0, hy0 = h.min(axis=0); hx1, hy1 = h.max(axis=0)
        near = (cand[idx, 0] >= hx0) & (cand[idx, 0] <= hx1) & (cand[idx, 1] >= hy0) & (cand[idx, 1] <= hy1)
        if near.any():
            in_hole = np.zeros(idx.size, dtype=bool)
            in_hole[near] = Path(h).contains_points(cand[idx[near]])
            idx = idx[~in_hole]
    labels[idx] = p["score"]
print(f"Point-in-polygon labelling took {time.time() - t0:.1f}s")

keep = labels >= 0
df = pd.DataFrame({"lon": cand[keep, 0], "lat": cand[keep, 1], "score": labels[keep]})
print(f"{len(df):,} labelled points (of {N_CANDIDATES:,} candidates)")
df["score"].value_counts().sort_index().rename("points_before_cap").to_frame()

In [ ]:
CLASS_NAMES = {0: "0 Water", 1: "1 Low (developed/barren)", 2: "2 Moderate (grass/crops)",
               3: "3 High (shrub/scrub)", 4: "4 Highest (forest)"}
COLORS = {0: "#2B83BA", 1: "#A6A6A6", 2: "#FDE047", 3: "#F28E2B", 4: "#D73027"}

# Optional cap per class: keeps training fast and reduces class imbalance
if MAX_PER_CLASS:
    df = pd.concat([g.sample(min(len(g), MAX_PER_CLASS), random_state=RANDOM_STATE)
                    for _, g in df.groupby("score")], ignore_index=True)

# Convert lon/lat to local kilometres so distances are isotropic for the RBF kernel
lat0 = df["lat"].mean()
df["x_km"] = (df["lon"] - df["lon"].mean()) * 111.32 * np.cos(np.radians(lat0))
df["y_km"] = (df["lat"] - lat0) * 110.57

print(df["score"].map(CLASS_NAMES).value_counts().sort_index())

fig, ax = plt.subplots(figsize=(8, 7))
for s in sorted(df["score"].unique()):
    d = df[df["score"] == s]
    ax.scatter(d["lon"], d["lat"], s=3, c=COLORS[s], label=CLASS_NAMES[s])
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
ax.set_title("Labelled sample points"); ax.set_aspect(1 / np.cos(np.radians(lat0)))
ax.legend(markerscale=4, loc="lower left", fontsize=8)
plt.show()

## 3. Train / test split

A random split overstates accuracy for spatial data because neighbouring points are almost identical. Two evaluations are therefore used:

* **Random hold-out (25%)**: the usual stratified split.
* **Spatial-block hold-out**: the area is divided into 15 km × 15 km blocks and whole blocks are held out. This tests how the model does on areas it has not seen near.

In [ ]:
X = df[["x_km", "y_km"]].to_numpy()
y = df["score"].to_numpy()

# 1) Random stratified hold-out
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)

# 2) Spatial-block hold-out (~25% of blocks)
BLOCK_KM = 15
block_id = (np.floor(df["x_km"] / BLOCK_KM).astype(int).astype(str) + "_" +
            np.floor(df["y_km"] / BLOCK_KM).astype(int).astype(str)).to_numpy()
blocks = np.unique(block_id)
test_blocks = rng.choice(blocks, size=max(1, int(0.25 * len(blocks))), replace=False)
is_test_block = np.isin(block_id, test_blocks)
X_train_b, y_train_b = X[~is_test_block], y[~is_test_block]
X_test_b, y_test_b = X[is_test_block], y[is_test_block]

print(f"Random split : train={len(X_train):,}  test={len(X_test):,}")
print(f"Block split  : train={len(X_train_b):,}  test={len(X_test_b):,}  ({len(blocks)} blocks, {len(test_blocks)} held out)")

## 4. SVM with hyperparameter tuning

An RBF-kernel `SVC` is used with standardised features and balanced class weights. `C` (margin softness) and `gamma` (kernel width, i.e. how local the decision regions are) are tuned with stratified 3-fold cross-validation.

In [ ]:
pipe = make_pipeline(StandardScaler(),
                     SVC(kernel="rbf", class_weight="balanced", decision_function_shape="ovr"))

param_grid = {
    "svc__C": [1, 10, 100],
    "svc__gamma": [10, 50, 200, 500],
}

grid = GridSearchCV(pipe, param_grid, scoring="balanced_accuracy",
                    cv=StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE),
                    n_jobs=-1, verbose=1)
t0 = time.time()
grid.fit(X_train, y_train)
print(f"\nTuning took {time.time() - t0:.1f}s")
print("Best params:", grid.best_params_)
print(f"Best CV balanced accuracy: {grid.best_score_:.3f}")

pd.DataFrame(grid.cv_results_)[["param_svc__C", "param_svc__gamma", "mean_test_score"]] \
  .pivot(index="param_svc__C", columns="param_svc__gamma", values="mean_test_score").round(3)

## 5. Evaluate on the random hold-out set

In [ ]:
best = grid.best_estimator_
y_pred = best.predict(X_test)
names = [CLASS_NAMES[c] for c in sorted(np.unique(y))]

print(f"Accuracy          : {accuracy_score(y_test, y_pred):.3f}")
print(f"Balanced accuracy : {balanced_accuracy_score(y_test, y_pred):.3f}")
print(f"Cohen's kappa     : {cohen_kappa_score(y_test, y_pred):.3f}\n")
print(classification_report(y_test, y_pred, target_names=names, digits=3))

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred), display_labels=[n.split()[0] for n in names]) \
    .plot(ax=axes[0], cmap="Blues", colorbar=False)
axes[0].set_title("Confusion matrix (counts)")
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred, normalize="true"), display_labels=[n.split()[0] for n in names]) \
    .plot(ax=axes[1], cmap="Blues", colorbar=False, values_format=".2f")
axes[1].set_title("Confusion matrix (row-normalised)")
plt.tight_layout(); plt.show()

## 6. Evaluate on spatially held-out blocks

In [ ]:
best_b = make_pipeline(StandardScaler(),
                       SVC(kernel="rbf", class_weight="balanced",
                           C=grid.best_params_["svc__C"], gamma=grid.best_params_["svc__gamma"]))
best_b.fit(X_train_b, y_train_b)
yb_pred = best_b.predict(X_test_b)

print(f"Accuracy          : {accuracy_score(y_test_b, yb_pred):.3f}")
print(f"Balanced accuracy : {balanced_accuracy_score(y_test_b, yb_pred):.3f}")
print(f"Cohen's kappa     : {cohen_kappa_score(y_test_b, yb_pred):.3f}\n")
present = sorted(np.unique(np.concatenate([y_test_b, yb_pred])))
print(classification_report(y_test_b, yb_pred, labels=present,
                            target_names=[CLASS_NAMES[c] for c in present], digits=3, zero_division=0))

## 7. Map the SVM's predicted classes

The model predicts a class for every cell of a grid across the county. Compare this with the true labelled points. The model is only meaningful inside the county, so cells farther than ~2 km from any sample point are masked.

In [ ]:
from sklearn.neighbors import NearestNeighbors

# Final model on all points for the map
final = make_pipeline(StandardScaler(),
                      SVC(kernel="rbf", class_weight="balanced",
                          C=grid.best_params_["svc__C"], gamma=grid.best_params_["svc__gamma"]))
final.fit(X, y)

gx = np.linspace(X[:, 0].min(), X[:, 0].max(), 350)
gy = np.linspace(X[:, 1].min(), X[:, 1].max(), 350)
GX, GY = np.meshgrid(gx, gy)
grid_pts = np.column_stack([GX.ravel(), GY.ravel()])

dist, _ = NearestNeighbors(n_neighbors=1).fit(X).kneighbors(grid_pts)
Z = final.predict(grid_pts).astype(float)
Z[dist.ravel() > 2.0] = np.nan
Z = Z.reshape(GX.shape)

cmap = ListedColormap([COLORS[c] for c in range(5)])
fig, axes = plt.subplots(1, 2, figsize=(15, 6.5), sharex=True, sharey=True)
axes[0].scatter(X[:, 0], X[:, 1], c=[COLORS[c] for c in y], s=3)
axes[0].set_title("True class (sample points)")
axes[1].pcolormesh(GX, GY, Z, cmap=cmap, vmin=-0.5, vmax=4.5, shading="auto")
axes[1].set_title("SVM predicted class (grid)")
for ax in axes:
    ax.set_aspect("equal"); ax.set_xlabel("x (km)")
axes[0].set_ylabel("y (km)")
fig.legend(handles=[Patch(color=COLORS[c], label=CLASS_NAMES[c]) for c in range(5)],
           loc="lower center", ncol=5, fontsize=9)
plt.tight_layout(rect=(0, 0.06, 1, 1)); plt.show()

## 8. Classify new points

Pass any longitude/latitude pairs to get predicted classes.

In [ ]:
lon_ref, lat_ref = df["lon"].mean(), lat0

def predict_points(lonlat):
    """lonlat: list of (lon, lat) tuples -> DataFrame with predicted class."""
    a = np.asarray(lonlat, dtype=float)
    xy = np.column_stack([(a[:, 0] - lon_ref) * 111.32 * np.cos(np.radians(lat_ref)),
                          (a[:, 1] - lat_ref) * 110.57])
    pred = final.predict(xy)
    return pd.DataFrame({"lon": a[:, 0], "lat": a[:, 1], "predicted_score": pred,
                         "predicted_class": [CLASS_NAMES[p] for p in pred]})

predict_points([(-118.2437, 34.0522),   # downtown Los Angeles
                (-118.0, 34.25),        # San Gabriel Mountains
                (-118.5, 33.6),         # offshore
                (-118.35, 33.38)])      # Santa Catalina Island

## Notes and next steps

* **Features:** With only coordinates, the SVM interpolates the map. Classification of a *new* area would need real predictors (elevation, slope, aspect, NDVI, distance to water, climate).
* **Scaling:** Training cost grows roughly with the square of the number of points; raise `MAX_PER_CLASS` for finer boundaries, but expect slower tuning.
* **Spatial vs. random scores:** The spatial-block result is the more honest estimate of generalisation.
* **Boundary precision:** Class borders are irregular at 90 m, so a smooth RBF boundary cannot capture the finest detail. Higher `gamma` follows borders more tightly but risks overfitting.